# RECI — Entrenamiento automático (MobileNetV2)

**Un solo clic — todo el pipeline de principio a fin.**

### Antes de empezar
1. Sube fotos nuevas a Drive:
   - `Mi unidad/RECI_dataset_propio/plastico/`
   - `Mi unidad/RECI_dataset_propio/vidrio/`
2. En Colab: **Entorno de ejecución → Cambiar tipo → GPU (T4)**
3. Menú **Ejecución → Ejecutar todo** (o `Ctrl+F9`)
4. Deja la pestaña abierta ~2–4 h. Al terminar, los archivos quedan en Drive.

### Qué hace este notebook (sin pausas manuales)
Organiza fotos nuevas → split 85/15 → entrena Fase 1 → fine-tuning Fase 2 → evalúa → exporta `.tflite`

### Salida
Guarda en `RECI_dataset_propio/runs/run_YYYYMMDD_HHMM/` para **no sobrescribir** el modelo anterior.

### Guía escrita (repo)
`docs/ENTRENAMIENTO_MODELO.md` — captura con `tomar_fotos.py`, subir a Drive, instalar el `.tflite` en RECI.

---
Notebook manual paso a paso: `RECI_entrenar_modelo.ipynb`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — edita solo si cambiaste la ruta de Drive
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime

DRIVE_BASE   = '/content/drive/MyDrive/RECI_dataset_propio'
RUTA_PLASTICO = f'{DRIVE_BASE}/plastico'
RUTA_VIDRIO   = f'{DRIVE_BASE}/vidrio'
DATASET_DIR   = f'{DRIVE_BASE}/dataset_organizado'

RUN_ID     = datetime.now().strftime('run_%Y%m%d_%H%M')
OUTPUT_DIR = f'{DRIVE_BASE}/runs/{RUN_ID}'

RANDOM_SEED = 42
SPLIT_TRAIN = 0.85
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS_FASE1 = 15
EPOCHS_FASE2 = 10
LR_FASE1    = 0.001
LR_FASE2    = 0.00005
EXTENSIONES_VALIDAS = ('.jpg', '.jpeg', '.png', '.webp')

print('Run ID   :', RUN_ID)
print('Salida   :', OUTPUT_DIR)

In [ ]:
print('\n' + '='*70)
print(' PASO 1/6 — Conectar Drive + verificar GPU')
print('='*70)

from google.colab import drive
drive.mount('/content/drive')

import os, random, shutil, json
import numpy as np
import tensorflow as tf

os.makedirs(OUTPUT_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(RANDOM_SEED)

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {gpus}')
if not gpus:
    print('⚠️  SIN GPU — será muy lento. Ve a Entorno → GPU antes de continuar.')
else:
    print('✓ GPU detectada')

In [ ]:
print('\n' + '='*70)
print(' PASO 2/6 — Organizar dataset (copiar fotos nuevas → train/val 85/15)')
print('='*70)

def contar_imagenes(ruta):
    if not os.path.isdir(ruta):
        return 0
    return len([f for f in os.listdir(ruta) if f.lower().endswith(EXTENSIONES_VALIDAS)])

for split in ['train', 'val']:
    for clase in ['plastico', 'vidrio']:
        os.makedirs(f'{DATASET_DIR}/{split}/{clase}', exist_ok=True)

CLASES = {'plastico': RUTA_PLASTICO, 'vidrio': RUTA_VIDRIO}
total_nuevas = 0

for clase, ruta_origen in CLASES.items():
    if not os.path.isdir(ruta_origen):
        raise FileNotFoundError(f'No existe {ruta_origen} — sube fotos a Drive primero.')

    ya_en_train = {f for f in os.listdir(f'{DATASET_DIR}/train/{clase}')
                   if f.lower().endswith(EXTENSIONES_VALIDAS)}
    ya_en_val   = {f for f in os.listdir(f'{DATASET_DIR}/val/{clase}')
                   if f.lower().endswith(EXTENSIONES_VALIDAS)}
    ya_copiadas = ya_en_train | ya_en_val

    todas = sorted(f for f in os.listdir(ruta_origen)
                 if f.lower().endswith(EXTENSIONES_VALIDAS))
    nuevas = [f for f in todas if f not in ya_copiadas]

    if nuevas:
        rng = random.Random(RANDOM_SEED)
        rng.shuffle(nuevas)
        corte = int(len(nuevas) * SPLIT_TRAIN)
        for img in nuevas[:corte]:
            shutil.copy(f'{ruta_origen}/{img}', f'{DATASET_DIR}/train/{clase}/{img}')
        for img in nuevas[corte:]:
            shutil.copy(f'{ruta_origen}/{img}', f'{DATASET_DIR}/val/{clase}/{img}')
        total_nuevas += len(nuevas)
        print(f'  {clase}: +{len(nuevas)} fotos nuevas copiadas')
    else:
        print(f'  {clase}: sin fotos nuevas ({len(todas)} en origen)')

print(f'\nFotos nuevas agregadas al split: {total_nuevas}')
print('\nResumen dataset organizado:')
stats = {}
for split in ['train', 'val']:
    stats[split] = {}
    for clase in ['plastico', 'vidrio']:
        n = contar_imagenes(f'{DATASET_DIR}/{split}/{clase}')
        stats[split][clase] = n
        print(f'  {split}/{clase}: {n}')
total = sum(stats[s][c] for s in stats for c in stats[s])
print(f'  TOTAL: {total} fotos')
if total == 0:
    raise ValueError('Dataset vacío — revisa las carpetas en Drive.')

In [ ]:
print('\n' + '='*70)
print(' PASO 3/6 — Preparar datos (augmentation + class weights)')
print('='*70)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', seed=RANDOM_SEED),
    tf.keras.layers.RandomRotation(0.1, seed=RANDOM_SEED),
    tf.keras.layers.RandomZoom(0.1, seed=RANDOM_SEED),
    tf.keras.layers.RandomBrightness(0.1, seed=RANDOM_SEED),
])

train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATASET_DIR}/train', image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='categorical',
    seed=RANDOM_SEED, shuffle=True)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATASET_DIR}/val', image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, label_mode='categorical',
    seed=RANDOM_SEED, shuffle=False)

CLASES_NOMBRES = train_ds.class_names
print('Clases:', CLASES_NOMBRES)

conteos_train = {c: contar_imagenes(f'{DATASET_DIR}/train/{c}') for c in CLASES_NOMBRES}
total_train = sum(conteos_train.values())
class_weight = {i: total_train / (len(CLASES_NOMBRES) * conteos_train[c])
                for i, c in enumerate(CLASES_NOMBRES)}
print('Conteos train:', conteos_train)
print('Class weights:', class_weight)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)

In [ ]:
print('\n' + '='*70)
print(' PASO 4/6 — Fase 1: entrenar capas nuevas (base MobileNetV2 congelada)')
print('='*70)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x       = base_model(x, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASES_NOMBRES), activation='softmax')(x)
model   = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(LR_FASE1),
              loss='categorical_crossentropy', metrics=['accuracy'])

ckpt1 = f'{OUTPUT_DIR}/mejor_modelo.keras'
cb1 = [
    tf.keras.callbacks.ModelCheckpoint(ckpt1, save_best_only=True, monitor='val_accuracy', verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                     restore_best_weights=True, verbose=1),
]

hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FASE1,
                  callbacks=cb1, class_weight=class_weight)
best1 = max(hist1.history['val_accuracy'])
print(f'\n✓ Fase 1 terminada — mejor val_accuracy: {best1:.1%}')

In [ ]:
print('\n' + '='*70)
print(' PASO 5/6 — Fase 2: fine-tuning (últimas 30 capas MobileNetV2)')
print('='*70)

model = tf.keras.models.load_model(ckpt1)

base = None
for layer in model.layers:
    if 'mobilenetv2' in layer.name.lower():
        base = layer
        break
if base is None:
    raise ValueError('No se encontró MobileNetV2 en el modelo')

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(LR_FASE2),
              loss='categorical_crossentropy', metrics=['accuracy'])

ckpt2 = f'{OUTPUT_DIR}/mejor_modelo_ft.keras'
cb2 = [
    tf.keras.callbacks.ModelCheckpoint(ckpt2, save_best_only=True, monitor='val_accuracy', verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                     restore_best_weights=True, verbose=1),
]

hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FASE2,
                  callbacks=cb2, class_weight=class_weight)
best2 = max(hist2.history['val_accuracy'])
print(f'\n✓ Fase 2 terminada — mejor val_accuracy: {best2:.1%}')

In [ ]:
print('\n' + '='*70)
print(' PASO 6/6 — Evaluar + exportar TFLite')
print('='*70)

model = tf.keras.models.load_model(ckpt2)
loss, accuracy = model.evaluate(val_ds)
print(f'\nPrecisión final: {accuracy:.1%}')
print(f'Loss final:      {loss:.4f}')

y_true, y_pred = [], []
for batch_x, batch_y in val_ds:
    probs = model.predict(batch_x, verbose=0)
    y_true.extend(np.argmax(batch_y.numpy(), axis=1))
    y_pred.extend(np.argmax(probs, axis=1))

cm = tf.math.confusion_matrix(
    np.array(y_true), np.array(y_pred), num_classes=len(CLASES_NOMBRES)).numpy()
print('\nMatriz de confusión (filas=real, cols=pred):')
print(cm)

print('\nMétricas por clase:')
metricas_clase = {}
for idx, clase in enumerate(CLASES_NOMBRES):
    tp = cm[idx, idx]; fp = cm[:, idx].sum() - tp; fn = cm[idx, :].sum() - tp
    p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0
    f1 = 2*p*r/(p+r) if p+r else 0
    print(f'  {clase:10s} precision={p:.3f} recall={r:.3f} f1={f1:.3f}')
    metricas_clase[clase] = {'precision': float(p), 'recall': float(r), 'f1': float(f1)}

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()

tflite_path = f'{OUTPUT_DIR}/model.tflite'
labels_path = f'{OUTPUT_DIR}/labels.txt'
with open(tflite_path, 'wb') as f:
    f.write(tflite_bytes)
with open(labels_path, 'w') as f:
    for i, c in enumerate(CLASES_NOMBRES):
        f.write(f'{i} {c}\n')

manifest = {
    'run_id': RUN_ID,
    'accuracy': float(accuracy),
    'loss': float(loss),
    'best_val_fase1': float(best1),
    'best_val_fase2': float(best2),
    'stats_dataset': stats,
    'class_weight': {str(k): v for k, v in class_weight.items()},
    'confusion_matrix': cm.tolist(),
    'metricas_por_clase': metricas_clase,
    'clases': CLASES_NOMBRES,
}
manifest_path = f'{OUTPUT_DIR}/entrenamiento_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

mb = os.path.getsize(tflite_path) / 1024 / 1024
print(f'\n✓ model.tflite  ({mb:.1f} MB) → {tflite_path}')
print(f'✓ labels.txt    → {labels_path}')
print(f'✓ manifest JSON → {manifest_path}')

if accuracy >= 0.90:
    print('\n🏆 Modelo listo — copia a RECI cuando verifiques:')
else:
    print('\n⚠️  Precisión < 90% — revisa fotos o agrega más vidrio antes de reemplazar el modelo.')
print(f'   cp {tflite_path} → model/model.tflite en tu Mac')

---
## (Opcional) Descargar al Mac

Ejecuta **solo esta celda** si quieres bajar los archivos. No hace falta para el entrenamiento.

También puedes descargarlos desde Google Drive en la carpeta `runs/run_.../`

In [ ]:
# OPCIONAL — no incluido en "Ejecutar todo" si prefieres solo Drive
# Descomenta las 3 líneas de abajo para descargar al Mac:

# from google.colab import files
# files.download(tflite_path)
# files.download(labels_path)

print('Descarga opcional — descomenta el código de arriba o usa Drive.')
print('Carpeta del run:', OUTPUT_DIR)